In [1]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, Span

In [2]:


# 1. Load and Process Data
df = pd.read_csv("results/output.csv")

# Aggregate by episode
eval_summary = df.groupby('episode').agg(
    total_reward=('reward', 'sum'),
    steps=('step', 'max'),
    final_status=('drone_status', 'last')
).reset_index()

# 2. Performance Distribution (Histogram)
# This shows how "reliable" the model is.
hist, edges = np.histogram(eval_summary['total_reward'], bins=10)
p1 = figure(title="Reward Reliability (Distribution)", 
            x_axis_label='Total Reward', y_axis_label='Frequency', height=350)
p1.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], 
        fill_color="#718dbf", line_color="white")

# 3. Efficiency vs. Outcome (Scatter Plot)
# Helps see if "fast" episodes lead to "crashes"
source = ColumnDataSource(eval_summary)
p2 = figure(title="Step Count vs Reward", 
            x_axis_label='Steps', y_axis_label='Total Reward', height=350)
p2.scatter(x='steps', y='total_reward', source=source, size=10, 
           color="#e84d60", alpha=0.6)

# 4. Success Rate Calculation
success_count = len(eval_summary[eval_summary['final_status'] == 3]) # Change string to match your env
success_rate = (success_count / len(eval_summary)) * 100

# 5. Display
print(f"Evaluation Complete. Success Rate: {success_rate}%")
output_file("model_evaluation.html")
show(gridplot([[p1, p2]]))

Evaluation Complete. Success Rate: 44.0%


In [3]:
import pandas as pd
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import cumsum
from math import pi

# 1. Load and Process Data
df = pd.read_csv("results/output.csv")

# Aggregate by episode for the line chart (Reward & Efficiency)
episode_summary = df.groupby('episode').agg(
    total_reward=('reward', 'sum'),
    steps=('step', 'max'),
    final_status=('drone_status', 'last')
).reset_index()

# Aggregate for Outcome Distribution (Pie Chart)
outcomes = episode_summary['final_status'].value_counts().reset_index()
outcomes.columns = ['status', 'count']
outcomes['angle'] = outcomes['count'] / outcomes['count'].sum() * 2 * pi
outcomes['color'] = ["#3182bd", "#9ecae1", "#deebf7", "#636363"][:len(outcomes)]

# 2. Create the Reward Trend Plot
source = ColumnDataSource(episode_summary)
p1 = figure(title="Total Reward per Episode", x_axis_label='Episode', 
            y_axis_label='Reward', width=600, height=400)

p1.line(x='episode', y='total_reward', source=source, line_width=2, color="#2ca02c")
p1.add_tools(HoverTool(tooltips=[("Episode", "@episode"), ("Reward", "@total_reward"), ("Status", "@final_status")]))

# 3. Create the Sample Efficiency Plot (Steps per Episode)
p2 = figure(title="Steps to Completion (Sample Efficiency)", x_axis_label='Episode', 
            y_axis_label='Steps', width=600, height=400)
p2.vbar(x='episode', top='steps', source=source, width=0.7, color="#1f77b4")

# 4. Create the Outcome Distribution (Pie Chart)
p3 = figure(height=400, title="Final Outcome Distribution", toolbar_location=None,
           tools="hover", tooltips="@status: @count", x_range=(-0.5, 1.0))

p3.wedge(x=0, y=1, radius=0.4, 
        start_angle=cumsum('angle', include_zero=True), end_angle=cumsum('angle'),
        line_color="white", fill_color='color', legend_field='status', source=outcomes)

p3.axis.axis_label = None
p3.axis.visible = False
p3.grid.grid_line_color = None

# 5. Layout and Output
output_file("drone_analysis.html")
show(column(p1, row(p2, p3)))

# Identify non-controlled states


In [21]:

df = pd.read_csv("results/output.csv")

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

episode_ends = df.groupby('episode')['drone_status'].last()
failed_episode_ids = episode_ends[episode_ends != 3].index.tolist()

failed_trajectories = df[df['episode'].isin(failed_episode_ids)]
last_moments = failed_trajectories.groupby('episode').tail(5)
print(last_moments[['episode', 'step', 'action', 'drone_status', 'network_has_password', 'password_list_has_password', 'target_port_vulnerable']])

      episode  step  action  drone_status  network_has_password  password_list_has_password  target_port_vulnerable
33          0    33       6             0                  True                       False                   False
34          0    34       6             0                  True                       False                   False
35          0    35       6             0                  True                       False                   False
36          0    36       6             0                  True                       False                   False
37          0    37       6             0                  True                       False                   False
80          2    34       1             0                  True                        True                    True
81          2    35       1             0                  True                        True                    True
82          2    36       1             0                  True         